# Export for the Site
This notebook packs everything the site needs into one JSON file. For every player it stores
his season line, each week he played, his rank as of each week, and every play that moved his
adjusted total. The site shows a position table with actual points, the point columns (Flag,
QB Fault, Drop, Replay, Short, and Gained), and adjusted points, and every row adds up: actual
plus the columns equals adjusted. The last cell puts it all into the page, index.html. Run it
after notebook 5 each week.

In [4]:
import json
import re
from datetime import date

import nflreadpy as nfl
import numpy as np
import pandas as pd

In [5]:
# ---------------- SETTINGS ----------------
CURRENT_SZN = 2026
POSITIONS = ["QB", "RB", "WR", "TE"]

# Which notebook 4 and 5 events go in which column on the site
COLUMNS = {
    "flag":     ["dpi", "penalty_nullified", "penalty_truncated"],
    "qb_fault": ["open_miss"],
    "drop":     ["drop"],
    "replay":   ["overturned_td", "replay_reversal"],
    # Goal-line fumbles: the lost TD (down_short_fumble) and the 2 points for the fumble itself
    "short":    ["down_short", "down_short_fumble", "fumble"],
    "gained":   ["td_transfer"],
}
# Hail Marys split by what happened: a caught Hail Mary TD is a gain, and a Hail Mary pick
# given back goes with the QB's missed throws
HAIL_MARY_GAIN, HAIL_MARY_PICK = "gained", "qb_fault"

COLUMN_ORDER = ["flag", "qb_fault", "drop", "replay", "short", "gained"]
LABELS = {
    "default": {"flag": "Flag", "qb_fault": "QB Fault", "drop": "Drop", "replay": "Replay",
                "short": "Short", "gained": "Gained"},
    # On the QB table, QB Fault is his own missed throws and Drop is his receivers' drops
    "QB": {"qb_fault": "Missed"},
}

In [6]:
# ---------------- LOAD ----------------
season = pd.read_parquet(f"data/season_ranks_{CURRENT_SZN}.parquet")      # notebook 5
weekly = pd.read_parquet(f"data/weekly_ranks_{CURRENT_SZN}.parquet")      # notebook 5
as_of = pd.read_parquet(f"data/ranks_by_week_{CURRENT_SZN}.parquet")      # notebook 5
luck = pd.read_parquet(f"data/luck_ledger_{CURRENT_SZN}.parquet")         # notebook 4
actual_ledger = pd.read_parquet(f"data/ledger_{CURRENT_SZN}.parquet")     # notebook 2
pbp = pd.read_parquet(f"data/pbp_{CURRENT_SZN}.parquet",
                      columns=["game_id", "play_id", "week", "qtr", "time", "desc", "posteam"])
official = pd.read_parquet(f"data/player_stats_{CURRENT_SZN}.parquet")
official = official[official["season_type"] == "REG"]

season = season[season["position_group"].isin(POSITIONS)]
weekly = weekly[weekly["position_group"].isin(POSITIONS)]
as_of = as_of[as_of["position_group"].isin(POSITIONS)]
through_week = int(weekly["week"].max())
print(f"Players: {len(season):,}   Player-weeks: {len(weekly):,}   Through week {through_week}")

Players: 535   Player-weeks: 3,096   Through week 11


## Every play that moved a player's adjusted total
This is the same set of plays notebook 5 added up: the luck ledger from notebook 4 plus the
goal-line fumbles from notebook 2 (skipping the ones replay already gave back). Each play gets
one row per player per column, with its points summed across stats.

In [8]:
fumbles = actual_ledger[actual_ledger["stat"] == "fumbles_lost"].copy()
replay_fumbles = luck[(luck["event"] == "replay_reversal") & (luck["stat"] == "fumbles_lost")]
replay_keys = set(zip(replay_fumbles["game_id"], replay_fumbles["play_id"], replay_fumbles["player_id"]))
is_replay = np.array([k in replay_keys for k in zip(fumbles["game_id"], fumbles["play_id"], fumbles["player_id"])], dtype=bool)
fumbles = fumbles[~is_replay]
goal_line = luck[luck["event"] == "down_short_fumble"]
goal_line_keys = set(zip(goal_line["game_id"], goal_line["play_id"], goal_line["player_id"]))
fumbles = fumbles[np.array([k in goal_line_keys for k in zip(fumbles["game_id"], fumbles["play_id"], fumbles["player_id"])], dtype=bool)]
fumbles = fumbles.assign(points=lambda d: -d["points"], event="fumble")

rows = pd.concat([
    luck[["game_id", "play_id", "player_id", "event", "points"]],
    fumbles[["game_id", "play_id", "player_id", "event", "points"]],
], ignore_index=True)

event_to_col = {e: c for c, events in COLUMNS.items() for e in events}
rows["column"] = rows["event"].map(event_to_col)
is_hail = rows["event"] == "hail_mary"
rows.loc[is_hail, "column"] = np.where(rows.loc[is_hail, "points"] < 0, HAIL_MARY_GAIN, HAIL_MARY_PICK)

unmapped = rows.loc[rows["column"].isna(), "event"].unique()
if len(unmapped):
    print("WARNING: events with no column:", unmapped)

plays = (rows.groupby(["game_id", "play_id", "player_id", "column"], as_index=False)
         .agg(points=("points", "sum"), events=("event", lambda s: sorted(set(s)))))
plays = plays.merge(pbp, on=["game_id", "play_id"], how="left")
plays["quarter"] = plays["qtr"].map(lambda q: "OT" if q >= 5 else f"Q{int(q)}")
plays = plays[plays["player_id"].isin(season["player_id"])]

# A play counts toward a column's play count if it moved points in that column. Target-only
# rows (0 points) stay out of the count.
plays["counts"] = plays["points"].round(2) != 0
print(f"Plays in the log: {plays['counts'].sum():,}")

Plays in the log: 4,405


## Column totals per player-week
The points and play counts for each column, rebuilt from the play log so the log and the
table can never disagree.

In [10]:
GAME = ["game_id", "player_id"]
pts = plays.pivot_table(index=GAME, columns="column", values="points", aggfunc="sum", fill_value=0)
cnt = plays[plays["counts"]].pivot_table(index=GAME, columns="column", values="points", aggfunc="count", fill_value=0)
pts = pts.reindex(columns=COLUMN_ORDER, fill_value=0).add_suffix("_pts")
cnt = cnt.reindex(columns=COLUMN_ORDER, fill_value=0).add_suffix("_plays")

# Notebook 5's per-event columns are replaced by the site columns
weekly = weekly.drop(columns=[c for c in weekly.columns if c.endswith(("_pts", "_plays"))])
weekly = weekly.merge(pts.reset_index(), on=GAME, how="left").merge(cnt.reset_index(), on=GAME, how="left")
new_cols = [f"{c}_pts" for c in COLUMN_ORDER] + [f"{c}_plays" for c in COLUMN_ORDER]
weekly[new_cols] = weekly[new_cols].fillna(0)

# Team and opponent for each game. Players with no official stat line that week (only almost
# points) get their team from the play-by-play.
game_team = official[["game_id", "player_id", "team", "opponent_team"]].drop_duplicates(GAME)
weekly = weekly.drop(columns="team").merge(game_team, on=GAME, how="left")
luck_team = luck.drop_duplicates(GAME).set_index(GAME)["posteam"]
missing = weekly["team"].isna()
weekly.loc[missing, "team"] = [luck_team.get(k) for k in zip(weekly.loc[missing, "game_id"], weekly.loc[missing, "player_id"])]

def opponent(game_id, team):
    away, home = game_id.split("_")[2:4]
    return home if team == away else away

weekly["home"] = [g.split("_")[3] == t for g, t in zip(weekly["game_id"], weekly["team"])]
weekly["opponent_team"] = weekly["opponent_team"].fillna(
    pd.Series([opponent(g, t) for g, t in zip(weekly["game_id"], weekly["team"])], index=weekly.index))

## Test 1: every row adds up
Actual plus the columns must equal notebook 5's adjusted (should-be) total for every player-week
and every season line. If this fails, an event is missing from COLUMNS or got counted twice.

In [12]:
week_sum = weekly["actual"] + weekly[[f"{c}_pts" for c in COLUMN_ORDER]].sum(axis=1)
bad_weeks = (week_sum - weekly["should_be"]).abs() > 0.01
print(f"Player-weeks that don't add up: {bad_weeks.sum()}")

season_cols = weekly.groupby("player_id")[[f"{c}_pts" for c in COLUMN_ORDER] + [f"{c}_plays" for c in COLUMN_ORDER]].sum()
season = season.drop(columns=[c for c in season.columns if c.endswith(("_pts", "_plays"))])
season = season.merge(season_cols.reset_index(), on="player_id", how="left")
season[new_cols] = season[new_cols].fillna(0)
season_sum = season["actual"] + season[[f"{c}_pts" for c in COLUMN_ORDER]].sum(axis=1)
bad_season = (season_sum - season["should_be"]).abs() > 0.01
print(f"Season lines that don't add up: {bad_season.sum()}")
assert not bad_weeks.any() and not bad_season.any()

Player-weeks that don't add up: 0
Season lines that don't add up: 0


## Build the JSON
Numbers are rounded to one decimal to keep the file small. Adjusted is the should-be total and
rank from notebook 5.

In [14]:
def r1(x):
    return round(float(x), 1)

def column_block(row):
    return {c: [r1(row[f"{c}_pts"]), int(row[f"{c}_plays"])] for c in COLUMN_ORDER}

# Headshots: ESPN's are sharper than the NFL's, so ESPN's is used whenever nflverse's player table
# has his ESPN ID. Otherwise it falls back to the NFL headshot from his most recent weekly stats.
nfl_headshots = (official.sort_values("week").dropna(subset=["headshot_url"])
                 .groupby("player_id")["headshot_url"].last().to_dict())
player_ids = nfl.load_players().to_pandas()[["gsis_id", "espn_id"]].dropna().drop_duplicates("gsis_id")
espn_ids = {g: str(int(float(e))) for g, e in zip(player_ids["gsis_id"], player_ids["espn_id"])}
ESPN_HEADSHOT = "https://a.espncdn.com/i/headshots/nfl/players/full/"
headshots = {pid: ESPN_HEADSHOT + espn_ids[pid] + ".png" if pid in espn_ids else nfl_headshots.get(pid)
             for pid in season["player_id"]}

# Team names and logos from nflverse's team table
teams = nfl.load_teams().to_pandas()
teams = {r["team_abbr"]: {"name": r["team_name"], "logo": r["team_logo_espn"]} for _, r in teams.iterrows()}

weeks_by_player = {pid: g.sort_values("week") for pid, g in weekly.groupby("player_id")}
as_of_by_player = {pid: g.sort_values("through_week") for pid, g in as_of.groupby("player_id")}
plays_by_player = {pid: g.sort_values(["week", "qtr", "play_id"]) for pid, g in plays[plays["counts"]].groupby("player_id")}

players_out = []
for _, p in season.sort_values(["position_group", "should_be_rank"]).iterrows():
    pid = p["player_id"]
    wk = weeks_by_player.get(pid, pd.DataFrame())
    players_out.append({
        "id": pid,
        "name": p["player_display_name"],
        "pos": p["position_group"],
        "team": p["team"],
        "headshot": headshots.get(pid),
        "games": int(p["games"]),
        "season": {
            "actual": r1(p["actual"]), "actual_rank": int(p["actual_rank"]),
            "adjusted": r1(p["should_be"]), "adjusted_rank": int(p["should_be_rank"]),
            "actual_pg": r1(p["actual_per_game"]), "adjusted_pg": r1(p["should_be_per_game"]),
            "cols": column_block(p),
        },
        "weeks": [{
            "week": int(w["week"]), "team": w["team"], "opp": w["opponent_team"], "home": bool(w["home"]),
            "actual": r1(w["actual"]), "actual_rank": int(w["actual_rank"]),
            "adjusted": r1(w["should_be"]), "adjusted_rank": int(w["should_be_rank"]),
            "cols": column_block(w),
        } for _, w in wk.iterrows()],
        "trend": [[int(a["through_week"]), int(a["actual_rank"]), int(a["should_be_rank"])]
                  for _, a in as_of_by_player.get(pid, pd.DataFrame()).iterrows()],
        "log": [{
            "week": int(l["week"]), "q": l["quarter"], "clock": l["time"], "col": l["column"],
            "pts": r1(l["points"]), "desc": re.sub(r"\s+", " ", str(l["desc"])).strip(),
        } for _, l in plays_by_player.get(pid, pd.DataFrame()).iterrows()],
    })

export = {
    "season": CURRENT_SZN,
    "through_week": through_week,
    "updated": date.today().isoformat(),
    "columns": COLUMN_ORDER,
    "labels": {pos: {**LABELS["default"], **LABELS.get(pos, {})} for pos in POSITIONS},
    "teams": teams,
    "players": players_out,
}

## Test 2: the JSON matches the tables
Reads the file back and spot-checks it against notebook 5's season table.

In [16]:
path = f"data/site_{CURRENT_SZN}.json"
with open(path, "w") as f:
    json.dump(export, f, separators=(",", ":"))

with open(path) as f:
    back = json.load(f)

by_id = {p["id"]: p for p in back["players"]}
check = season.set_index("player_id")
off = [pid for pid, p in by_id.items()
       if abs(p["season"]["actual"] - check.loc[pid, "actual"]) > 0.06
       or abs(p["season"]["adjusted"] - check.loc[pid, "should_be"]) > 0.06
       or p["season"]["adjusted_rank"] != check.loc[pid, "should_be_rank"]]
log_off = [pid for pid, p in by_id.items()
           if abs(sum(l["pts"] for l in p["log"]) - sum(v[0] for v in p["season"]["cols"].values())) > 0.05 * max(1, len(p["log"]))]
print(f"Players in file: {len(by_id):,}   Season lines off: {len(off)}   Logs that don't match their columns: {len(log_off)}")
print(f"File size: {len(json.dumps(back, separators=(',', ':'))) / 1e6:.2f} MB   Saved {path}")

Players in file: 535   Season lines off: 0   Logs that don't match their columns: 0
File size: 1.76 MB   Saved data/site_2026.json


## Build the page
Puts this week's data into the page template and saves the page as WFR/site/almost_points/index.html,
this tab's own folder inside the Wave Fantasy Report site. The template lives in FantasyLuck/templates/
so it never gets published. Open the page by double-clicking.

The page gets a packed copy of the data: numbers in short arrays instead of named fields, headshot
links without their shared start, and the play logs in their own block that the page only reads
when a player page is opened. Nothing is dropped, so the page shows exactly what's in the JSON.

In [18]:
HEADSHOT_PREFIX = "https://static.www.nfl.com/image/upload/f_auto,q_auto/league/"

def pack_cols(cols):
    return [v for c in COLUMN_ORDER for v in cols[c]]

def short_headshot(url):
    # ESPN headshots are stored as "E:" plus the ESPN ID, and NFL ones without their shared start
    if not url:
        return None
    if url.startswith(ESPN_HEADSHOT):
        return "E:" + url[len(ESPN_HEADSHOT):-len(".png")]
    return url[len(HEADSHOT_PREFIX):] if url.startswith(HEADSHOT_PREFIX) else url

col_index = {c: i for i, c in enumerate(COLUMN_ORDER)}
packed = {k: v for k, v in export.items() if k != "players"}
packed["headshot_prefix"] = HEADSHOT_PREFIX
packed["espn_headshot_prefix"] = ESPN_HEADSHOT
packed["players"] = []
logs = {}
for p in export["players"]:
    s = p["season"]
    packed["players"].append([
        p["id"], p["name"], p["pos"], p["team"], short_headshot(p["headshot"]), p["games"],
        [s["actual"], s["actual_rank"], s["adjusted"], s["adjusted_rank"], s["actual_pg"], s["adjusted_pg"]] + pack_cols(s["cols"]),
        [[w["week"], w["team"], w["opp"], int(w["home"]), w["actual"], w["actual_rank"], w["adjusted"], w["adjusted_rank"]]
         + pack_cols(w["cols"]) for w in p["weeks"]],
        p["trend"],
    ])
    if p["log"]:
        # The clock at the start of each description is dropped, since the page shows it already
        logs[p["id"]] = [[l["week"], l["q"], l["clock"], col_index[l["col"]], l["pts"],
                          re.sub(r"^\(\d*:\d+\)\s*", "", l["desc"])] for l in p["log"]]

def to_js(obj):
    # "</" is escaped so a play description can never close the script tag early
    return json.dumps(obj, separators=(",", ":"), ensure_ascii=False).replace("</", "<\\/")

with open("templates/almost_points_template.html", encoding="utf-8") as f:
    template = f.read()
assert "/*__DATA__*/null" in template and "/*__LOGS__*/" in template, "The template is missing a placeholder"
page = template.replace("/*__DATA__*/null", to_js(packed)).replace("/*__LOGS__*/", to_js(logs))

# ../ steps up out of FantasyLuck into WFR, then into the site folder
with open("../site/almost_points/index.html", "w", encoding="utf-8") as f:
    f.write(page)
print(f"Saved site/almost_points/index.html ({len(page.encode('utf-8')) / 1e6:.2f} MB), through week {export['through_week']}")

Saved site/almost_points/index.html (0.95 MB), through week 11
